# XGBoost + NN Predictor Features — MLflow Experiment


In [1]:
import os
import json
import tempfile
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import xgboost as xgb
import mlflow
import mlflow.xgboost

from pathlib import Path
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score,
    confusion_matrix, classification_report,
    log_loss, brier_score_loss, balanced_accuracy_score,
    matthews_corrcoef,
)
from sklearn.calibration import calibration_curve

from Pipeline.pipeline import ForexDataLoader, ForexPipeline

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## Configuration

Edit `DATA_CFG`, `NN_FEATURES_CFG`, and `XGB_PARAMS` here.

**`DATA_CFG`** — pipeline parameters; also controls the shared filename tags for selected NN feature parquets
(`norm_method`, `weekends`, `fracdiff_d`, `scaling`, and `years`).

**`NN_FEATURES_CFG`** — provider-specific switches and inference parameters for Chronos, Kronos,
and TimesFM. Enable any one provider, any pair, or all three. Do **not** duplicate shared
normalization/weekend/scaling/year settings here; those are read from `DATA_CFG`.


In [ ]:
DATA_CFG = {
    "pair":             "EURUSD",
    "years":            [2023],
    "timeframe":        "H1",
    "weekends":         "filled",   # "filled" | "nogap" | "gaps"
    # normalization: "log_returns" | "fracdiff" | "raw"
    "norm_method":      "fracdiff",
    # target: "lag" (direction_1/5/15) | "triple_barrier" (tb_label)
    "target_type":      "triple_barrier",
    # target column: "tb_label" for triple_barrier | "direction_1/5/15" for lag
    "target_col":       "tb_label",
    "lags":             [1, 2, 5, 10],
    "target_horizons":  [1, 5, 15],  # pipeline target horizons used as features
    "gap_bars":         50,
    # scaling: "global" (ForexScaler) | "rolling" (RollingScaler) | "none"
    "scaling":          "none",
    "window_size":      500,         # only used if scaling="rolling"
    "fracdiff_d":       0.3,
    "threshold":        6 * 1e-4,
    "k_up":             1.0,
    "k_down":           1.0,
    "horizon_bars":     10,          # target horizon to predict
    "barrier_price":    "hl",
    "barrier_norm_method": "raw",
}

# NN feature params. Keep norm_method / weekends / scaling / years in DATA_CFG.
# Enable any one provider, any pair of providers, or all three.
NN_FEATURES_CFG = {
    "chronos": {
        "enabled":        True,
        "context_length": 504,
        "calc_interval":  5,
        "horizons":       [5, 10, 15, 20],
        "percentiles":    [0.05, 0.25, 0.50, 0.75, 0.95],
        "use_staleness":  False,
    },
    "kronos": {
        "enabled":        True,
        "context_length": 512,
        "calc_interval":  5,
        "horizons":       [5, 10, 15, 20],
        # n_samples == 1: OHLC/spread columns; n_samples > 1: quantile columns.
        "n_samples":      1,
        "percentiles":    [0.05, 0.25, 0.50, 0.75, 0.95],
        "temperature":    1.0,
        "top_p":          0.9,
        "use_staleness":  False,
    },
    "timesfm": {
        "enabled":        True,
        "context_length": 512,
        "calc_interval":  5,
        "horizons":       [5, 10, 15, 20],
        # TimesFM supports fixed levels: 0.0 mean, then 0.1 ... 0.9 quantiles.
        "percentiles":    [0.1, 0.3, 0.5, 0.7, 0.9],
        "use_staleness":  False,
    },
}

XGB_PARAMS = {
    "num_class":            3,
    "n_estimators":         300,
    "max_depth":            5,
    "learning_rate":        0.05,
    "subsample":            0.8,
    "colsample_bytree":     0.8,
    "min_child_weight":     50,
    "gamma":                0.1,
    "reg_alpha":            0.1,
    "reg_lambda":           1.0,
    "early_stopping_rounds": 20,
    "objective":            "multi:softprob",  # "binary:logistic" for lag targets
    "eval_metric":          "mlogloss",        # "logloss" for binary
    "random_state":         42,
    "n_jobs":               -1,
}

MLFLOW_DB       = f"sqlite:///{os.path.abspath('mlflow.db')}"
EXPERIMENT_NAME = "xgboost_nns_forex"

enabled_nn_sources = [name for name, cfg in NN_FEATURES_CFG.items() if cfg.get("enabled", False)]
assert enabled_nn_sources, "Enable at least one provider in NN_FEATURES_CFG."

print(f"DATA_CFG       : {json.dumps(DATA_CFG, indent=2)}")
print(f"NN_FEATURES_CFG: {json.dumps(NN_FEATURES_CFG, indent=2)}")
print(f"Enabled NN     : {enabled_nn_sources}")
print(f"XGB_PARAMS     : {json.dumps(XGB_PARAMS, indent=2)}")
print(f"MLflow DB      : {MLFLOW_DB}")


DATA_CFG   : {
  "pair": "EURUSD",
  "years": [
    2023
  ],
  "timeframe": "H1",
  "weekends": "filled",
  "norm_method": "fracdiff",
  "target_type": "triple_barrier",
  "target_col": "tb_label",
  "lags": [
    1,
    2,
    5,
    10
  ],
  "target_horizons": [
    1,
    5,
    15
  ],
  "gap_bars": 50,
  "scaling": "none",
  "window_size": 500,
  "fracdiff_d": 0.3,
  "threshold": 0.0006000000000000001,
  "k_up": 2.0,
  "k_down": 1.0,
  "horizon_bars": 10
}
CHRONOS_CFG: {
  "context_length": 504,
  "calc_interval": 10,
  "horizons": [
    5,
    10,
    15,
    20
  ],
  "use_staleness": false
}
XGB_PARAMS : {
  "num_class": 3,
  "n_estimators": 300,
  "max_depth": 5,
  "learning_rate": 0.05,
  "subsample": 0.8,
  "colsample_bytree": 0.8,
  "min_child_weight": 5,
  "gamma": 0.1,
  "reg_alpha": 0.1,
  "reg_lambda": 1.0,
  "early_stopping_rounds": 20,
  "objective": "multi:softprob",
  "eval_metric": "mlogloss",
  "random_state": 42,
  "n_jobs": -1
}
MLflow DB  : sqlite:////home/an

## 0 — Resolve NN Feature Parquet Paths


In [ ]:
def _norm_tag(data_cfg: dict) -> str:
    nm = data_cfg["norm_method"]
    if nm == "log_returns":
        return "logret"
    if nm == "fracdiff":
        return f"fdiff{data_cfg['fracdiff_d']}"
    return "raw"


def _h_tag(cfg: dict) -> str:
    return "h" + "-".join(str(h) for h in sorted(set(cfg["horizons"])))


def _scale_tag(data_cfg: dict) -> str:
    scaling = data_cfg["scaling"]
    if scaling == "none":
        return "snone"
    if scaling == "rolling":
        return f"sroll{data_cfg['window_size']}"
    return "sglob"


def _year_tag(data_cfg: dict) -> str:
    years = data_cfg.get("years")
    if years:
        return "_".join(str(y) for y in sorted(years))
    start = data_cfg.get("start")
    end = data_cfg.get("end")
    if start or end:
        return f"{(start or '').replace('-', '')}-{(end or '').replace('-', '')}"
    return "all"


def _build_nn_fname(source: str, data_cfg: dict, cfg: dict) -> str:
    pair = data_cfg["pair"]
    tf = data_cfg["timeframe"]
    ctx = cfg["context_length"]
    interval = cfg["calc_interval"]
    base = f"{pair}_{tf}"
    suffix = (
        f"_ctx{ctx}_int{interval}_{_h_tag(cfg)}"
        f"_{_norm_tag(data_cfg)}_w{data_cfg['weekends']}"
        f"_{_scale_tag(data_cfg)}_{_year_tag(data_cfg)}.parquet"
    )

    if source == "chronos":
        return f"{base}{suffix}"
    if source == "timesfm":
        return f"{base}_tfm{suffix}"
    if source == "kronos":
        sample_tag = f"_s{cfg['n_samples']}" if cfg.get("n_samples", 1) > 1 else ""
        return (
            f"{base}_kron_ctx{ctx}_int{interval}_{_h_tag(cfg)}"
            f"_{_norm_tag(data_cfg)}_w{data_cfg['weekends']}"
            f"_{_scale_tag(data_cfg)}{sample_tag}_{_year_tag(data_cfg)}.parquet"
        )
    raise ValueError(f"Unknown NN feature source: {source}")


def _generate_hint(source: str, data_cfg: dict, cfg: dict) -> str:
    imports = {
        "chronos": "from Chronos.chronos_features import generate",
        "kronos": "from Kronos.kronos_features import generate",
        "timesfm": "from Timeseriesfm.timesfm_features import generate",
    }
    args = {
        "pair": data_cfg["pair"],
        "years": data_cfg["years"],
        "timeframe": data_cfg["timeframe"],
        "context_length": cfg["context_length"],
        "calc_interval": cfg["calc_interval"],
        "horizons": cfg["horizons"],
        "norm_method": data_cfg["norm_method"],
        "fracdiff_d": data_cfg["fracdiff_d"],
        "threshold": data_cfg["threshold"],
        "weekends": data_cfg["weekends"],
        "scaling": data_cfg["scaling"],
        "scaling_window": data_cfg["window_size"],
    }
    if cfg.get("percentiles") is not None:
        args["percentiles"] = cfg["percentiles"]
    if source == "kronos":
        args["n_samples"] = cfg["n_samples"]
        args["temperature"] = cfg["temperature"]
        args["top_p"] = cfg["top_p"]
    rendered = ",\n    ".join(f"{k}={v!r}" for k, v in args.items())
    return f"{imports[source]}\ngenerate(\n    {rendered},\n)"


nn_feature_paths = {}
for source, cfg in NN_FEATURES_CFG.items():
    if not cfg.get("enabled", False):
        continue
    fname = _build_nn_fname(source, DATA_CFG, cfg)
    path = Path("featdata") / fname
    nn_feature_paths[source] = {"fname": fname, "path": path}
    print(f"{source:8s} expected parquet: {path}")
    if not path.exists():
        raise FileNotFoundError(
            f"\n\n{source} parquet not found: {path}\n"
            f"This notebook is load-only. Generate it first with:\n"
            f"{_generate_hint(source, DATA_CFG, cfg)}"
        )
    _probe = pd.read_parquet(path)
    print(f"  rows    : {len(_probe):,}")
    print(f"  columns : {list(_probe.columns)}")
    print(f"  range   : {_probe.index.min()} -> {_probe.index.max()}")
    del _probe


## 1 — Load Raw M1 Data

In [ ]:
loader = ForexDataLoader()
df_m1  = loader.load_and_merge(
    "histdata/",
    pair=DATA_CFG["pair"],
    years=DATA_CFG["years"],
    weekends=DATA_CFG["weekends"],
)

print(f"Raw M1 shape : {df_m1.shape}")
print(f"Date range   : {df_m1.index.min()}  →  {df_m1.index.max()}")
df_m1.head(3)

## 2 — Run Pipeline

In [ ]:
pipeline = ForexPipeline(
    lags            = DATA_CFG["lags"],
    target_horizons = DATA_CFG["target_horizons"],
    gap_bars        = DATA_CFG["gap_bars"],
    scaling         = DATA_CFG["scaling"],
    window_size     = DATA_CFG["window_size"],
    norm_method     = DATA_CFG["norm_method"],
    fracdiff_d      = DATA_CFG["fracdiff_d"],
    target_type     = DATA_CFG["target_type"],
    k_up            = DATA_CFG["k_up"],
    k_down          = DATA_CFG["k_down"],
    horizon_bars    = DATA_CFG["horizon_bars"],
    barrier_price   = DATA_CFG.get("barrier_price", "hl"),
    barrier_norm_method = DATA_CFG.get("barrier_norm_method", "raw"),
    threshold       = DATA_CFG["threshold"],
    weekends        = DATA_CFG["weekends"],
)

results      = pipeline.run(df_m1, timeframe=DATA_CFG["timeframe"])
feature_cols = results["feature_cols"]
target_col   = DATA_CFG["target_col"]

for split in ["train", "val", "test"]:
    df_s = results[split]
    print(f"{split:5s}: {len(df_s):6d} rows  "
          f"{df_s.index.min().date()} → {df_s.index.max().date()}")

print(f"\nPipeline features ({len(feature_cols)}): {feature_cols}")

## 3 — Load & Join NN Predictor Features


In [ ]:
import re as _re


def _pname(p: float, prefix: str = "q") -> str:
    return f"{prefix}{int(round(p * 100)):02d}"


def _select_source_columns(source: str, raw: pd.DataFrame, cfg: dict) -> list[str]:
    horizons = sorted(set(cfg["horizons"]))
    wanted_h = set(horizons)

    if source == "kronos" and cfg.get("n_samples", 1) == 1:
        expected = [
            f"{name}_h{h}"
            for h in horizons
            for name in ["close", "high", "low", "spread"]
        ]
    else:
        prefix = "tfm_q" if source == "timesfm" else "q"
        expected = [
            f"{_pname(p, prefix)}_h{h}"
            for h in horizons
            for p in sorted(set(cfg["percentiles"]))
        ]

    missing = [col for col in expected if col not in raw.columns]
    if missing:
        raise KeyError(
            f"{source} parquet is missing expected columns: {missing}. "
            f"Available columns: {list(raw.columns)}"
        )

    selected = expected.copy()
    if cfg.get("use_staleness", False):
        if "staleness" not in raw.columns:
            raise KeyError(f"{source} parquet does not contain staleness.")
        selected.append("staleness")

    leaked = [col for col in selected if col == "run_id"]
    assert not leaked, "run_id should remain metadata and not enter model features."
    return selected


nn_feature_frames = {}
nn_feature_cols_by_source = {}
nn_feature_cols = []
nn_feature_provider_by_col = {}

for source, cfg in NN_FEATURES_CFG.items():
    if not cfg.get("enabled", False):
        continue

    raw = pd.read_parquet(nn_feature_paths[source]["path"])
    source_cols = _select_source_columns(source, raw, cfg)
    prefixed = raw[source_cols].rename(columns={col: f"{source}_{col}" for col in source_cols})

    nn_feature_frames[source] = prefixed
    nn_feature_cols_by_source[source] = list(prefixed.columns)
    nn_feature_cols.extend(prefixed.columns)
    nn_feature_provider_by_col.update({col: source for col in prefixed.columns})

    print(f"{source:8s} selected columns ({len(source_cols)}): {source_cols}")
    print(f"{source:8s} prefixed columns ({len(prefixed.columns)}): {list(prefixed.columns)}")

if len(nn_feature_cols) != len(set(nn_feature_cols)):
    duplicates = pd.Series(nn_feature_cols).value_counts()
    raise ValueError(f"Duplicate NN feature names after prefixing: {duplicates[duplicates > 1].to_dict()}")

splits_extended = {}
for split_name in ["train", "val", "test"]:
    df = results[split_name].copy()
    n_before = len(df)
    for source, frame in nn_feature_frames.items():
        df = df.join(frame, how="left")
    df = df.dropna(subset=nn_feature_cols)
    n_dropped = n_before - len(df)
    splits_extended[split_name] = df
    print(f"{split_name:5s}: {len(df):6d} rows  "
          f"({n_dropped} dropped — no selected NN feature coverage)  "
          f"{df.index.min().date()} -> {df.index.max().date()}")

feature_cols_ext = feature_cols + nn_feature_cols
ENABLED_NN_LABEL = "+".join(nn_feature_frames.keys()).upper()
print(f"\nExtended feature set ({len(feature_cols_ext)}): "
      f"{len(feature_cols)} pipeline + {len(nn_feature_cols)} NN")
for source, cols in nn_feature_cols_by_source.items():
    print(f"  {source:8s}: {len(cols)} features")
print(f"Enabled NN label: {ENABLED_NN_LABEL}")


## 4 — Extract X / y Arrays

In [ ]:
# Class mapping after triple_barrier +1 remap:
#   0 = short hit   (tb_label = -1 : price hit the lower barrier)
#   1 = neutral     (tb_label =  0 : time expired, no barrier hit)
#   2 = long hit    (tb_label = +1 : price hit the upper barrier)
CLASS_LABELS = {0: "short", 1: "neutral", 2: "long"}

def extract_xy(splits_ext, split, target_col, feature_cols_ext):
    X, y = pipeline.get_xy(splits_ext[split], target_col, feature_cols_ext)
    if target_col == "tb_label":
        y = (y + 1).astype(int)  # -1→0, 0→1, 1→2
    return X, y.astype(int)

X_train, y_train = extract_xy(splits_extended, "train", target_col, feature_cols_ext)
X_val,   y_val   = extract_xy(splits_extended, "val",   target_col, feature_cols_ext)
X_test,  y_test  = extract_xy(splits_extended, "test",  target_col, feature_cols_ext)

print("Class mapping (triple_barrier):")
for cls, lbl in CLASS_LABELS.items():
    print(f"  {cls} = {lbl}")
print()

for name, y in [("train", y_train), ("val", y_val), ("test", y_test)]:
    total = len(y)
    unique, counts = np.unique(y, return_counts=True)
    dist_str = "  ".join(
        f"{CLASS_LABELS[int(k)]}({int(k)}): {int(v):,} ({int(v)/total:.1%})"
        for k, v in zip(unique, counts)
    )
    print(f"{name:5s}: {total:,} samples  |  {dist_str}")

## 5 — Train XGBoost + Log Everything to MLflow

In [ ]:
mlflow.set_tracking_uri(MLFLOW_DB)
mlflow.set_experiment(EXPERIMENT_NAME)

run_name = (
    f"{DATA_CFG['pair']}_{DATA_CFG['timeframe']}_"
    f"{DATA_CFG['norm_method']}_{DATA_CFG['target_col']}_"
    f"{ENABLED_NN_LABEL.lower()}"
)

with mlflow.start_run(run_name=run_name) as run:
    RUN_ID = run.info.run_id

    # ── Parameters ───────────────────────────────────────────────────────────
    mlflow.log_params({f"data_{k}": str(v) for k, v in DATA_CFG.items()})
    mlflow.log_params({f"xgb_{k}": v for k, v in XGB_PARAMS.items()})
    for source, cfg in NN_FEATURES_CFG.items():
        if not cfg.get("enabled", False):
            mlflow.log_param(f"{source}_enabled", False)
            continue
        mlflow.log_params({f"{source}_{k}": str(v) for k, v in cfg.items()})
        mlflow.log_param(f"{source}_file", nn_feature_paths[source]["fname"])
        mlflow.log_param(f"n_{source}_features", len(nn_feature_cols_by_source[source]))

    mlflow.log_param("enabled_nn_sources", ",".join(nn_feature_frames.keys()))
    mlflow.log_param("n_pipeline_features", len(feature_cols))
    mlflow.log_param("n_nn_features", len(nn_feature_cols))
    mlflow.log_param("n_features_total", len(feature_cols_ext))

    mlflow.log_params({
        "raw_m1_start":  str(df_m1.index.min()),
        "raw_m1_end":    str(df_m1.index.max()),
        "train_start":   str(splits_extended["train"].index.min()),
        "train_end":     str(splits_extended["train"].index.max()),
        "val_start":     str(splits_extended["val"].index.min()),
        "val_end":       str(splits_extended["val"].index.max()),
        "test_start":    str(splits_extended["test"].index.min()),
        "test_end":      str(splits_extended["test"].index.max()),
        "train_samples": len(X_train),
        "val_samples":   len(X_val),
        "test_samples":  len(X_test),
        "train_long_rate": float((y_train == 2).mean()),
        "val_long_rate":   float((y_val   == 2).mean()),
        "test_long_rate":  float((y_test  == 2).mean()),
    })

    mlflow.log_dict({
        "pipeline_features": feature_cols,
        "nn_features_by_source": nn_feature_cols_by_source,
        "features": feature_cols_ext,
    }, "features.json")

    # ── Train ─────────────────────────────────────────────────────────────────
    model = xgb.XGBClassifier(**XGB_PARAMS)
    model.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        verbose=False,
    )

    mlflow.log_param("best_iteration", model.best_iteration)
    mlflow.xgboost.log_model(model, "model")

    # ── Metrics per split ─────────────────────────────────────────────────────
    # proba shape: (n, 3)  ->  [P(short/0), P(neutral/1), P(long/2)]
    split_metrics = {}
    for sname, X, y in [("train", X_train, y_train),
                         ("val",   X_val,   y_val),
                         ("test",  X_test,  y_test)]:
        proba      = model.predict_proba(X)          # (n, 3)
        proba_long = proba[:, 2]                     # P(long hit) for binary metrics
        y_long     = (y == 2).astype(int)
        pred       = model.predict(X)
        m = {
            "auc":           roc_auc_score(y, proba, multi_class="ovr", average="macro"),
            "auc_long":      roc_auc_score(y_long, proba_long),
            "avg_precision": average_precision_score(y, proba, average="macro"),
            "logloss":       log_loss(y, proba),
            "brier_long":    brier_score_loss(y_long, proba_long),
            "f1":            f1_score(y, pred, average="macro", zero_division=0),
            "precision":     precision_score(y, pred, average="macro", zero_division=0),
            "recall":        recall_score(y, pred, average="macro", zero_division=0),
            "balanced_acc":  balanced_accuracy_score(y, pred),
            "mcc":           matthews_corrcoef(y, pred),
        }
        split_metrics[sname] = m
        mlflow.log_metrics({f"{sname}_{k}": v for k, v in m.items()})

    print(f"Run ID   : {RUN_ID}")
    print(f"Best iter: {model.best_iteration}")
    print(pd.DataFrame(split_metrics).T.round(4))


## 6 — Analysis: ROC Curves (train / val / test)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

colors = {"train": "steelblue", "val": "darkorange", "test": "seagreen"}
styles = {"train": "--",        "val": "-",          "test": "-"}

# ROC for "long hit" class (class 2) — most actionable signal
for sname, X, y in [("train", X_train, y_train),
                     ("val",   X_val,   y_val),
                     ("test",  X_test,  y_test)]:
    proba_long = model.predict_proba(X)[:, 2]
    y_long     = (y == 2).astype(int)
    fpr, tpr, _ = roc_curve(y_long, proba_long)
    auc        = roc_auc_score(y_long, proba_long)
    ax.plot(fpr, tpr, ls=styles[sname], color=colors[sname],
            label=f"{sname}  AUC = {auc:.4f}", lw=2)

ax.plot([0, 1], [0, 1], 'k:', lw=1, label="random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title(f"ROC Curve (long hit) — {DATA_CFG['pair']} {DATA_CFG['timeframe']} ({DATA_CFG['target_col']}) + {ENABLED_NN_LABEL}")
ax.legend(loc="lower right")
plt.tight_layout()

with mlflow.start_run(run_id=RUN_ID):
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "roc_curve.png")
        fig.savefig(path)
        mlflow.log_artifact(path, "plots")

plt.show()

## 7 — Analysis: Precision-Recall Curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

# PR for "long hit" class (class 2)
for sname, X, y in [("val",  X_val,  y_val),
                     ("test", X_test, y_test)]:
    proba_long = model.predict_proba(X)[:, 2]
    y_long     = (y == 2).astype(int)
    prec, rec, _ = precision_recall_curve(y_long, proba_long)
    ap         = average_precision_score(y_long, proba_long)
    ax.plot(rec, prec, color=colors[sname],
            label=f"{sname}  AP = {ap:.4f}", lw=2)

# No-skill baseline (long-hit base rate)
for sname, y in [("val", y_val), ("test", y_test)]:
    ax.axhline((y == 2).mean(), ls=':', color=colors[sname], alpha=0.5)

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title(f"Precision-Recall (long hit) — {DATA_CFG['pair']} {DATA_CFG['timeframe']} + {ENABLED_NN_LABEL}")
ax.legend()
plt.tight_layout()

with mlflow.start_run(run_id=RUN_ID):
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "pr_curve.png")
        fig.savefig(path)
        mlflow.log_artifact(path, "plots")

plt.show()

## 8 — Analysis: Confusion Matrices (val + test)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm_labels     = [f"{CLASS_LABELS[i]} ({i})" for i in range(3)]

for ax, (sname, X, y) in zip(axes, [("val", X_val, y_val), ("test", X_test, y_test)]):
    pred    = model.predict(X)
    cm      = confusion_matrix(y, pred, labels=[0, 1, 2])
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    annot   = np.array([[f"{cm[i,j]}\n({cm_norm[i,j]:.1%})" for j in range(3)] for i in range(3)])
    sns.heatmap(cm_norm, annot=annot, fmt='', cmap='Blues', ax=ax,
                xticklabels=[f"pred {l}" for l in cm_labels],
                yticklabels=[f"true {l}" for l in cm_labels],
                vmin=0, vmax=1, linewidths=0.5)
    ax.set_title(f"Confusion Matrix — {sname}")

plt.tight_layout()

with mlflow.start_run(run_id=RUN_ID):
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "confusion_matrices.png")
        fig.savefig(path)
        mlflow.log_artifact(path, "plots")

plt.show()

## 9 — Analysis: Classification Report

In [ ]:
report_frames = []
_target_names = [CLASS_LABELS[i] for i in range(3)]   # ["short", "neutral", "long"]

for sname, X, y in [("val", X_val, y_val), ("test", X_test, y_test)]:
    pred = model.predict(X)

    # Class distribution
    total = len(y)
    unique, counts = np.unique(y, return_counts=True)
    dist = {CLASS_LABELS[int(k)]: int(v) for k, v in zip(unique, counts)}
    print(f"\n── {sname}  (n={total:,}) ──")
    print("  Class counts: " + "  ".join(f"{lbl}: {cnt:,} ({cnt/total:.1%})"
                                          for lbl, cnt in dist.items()))

    print(classification_report(y, pred, target_names=_target_names))

    rpt    = classification_report(y, pred, target_names=_target_names, output_dict=True)
    df_rpt = pd.DataFrame(rpt).T.round(4)
    df_rpt.insert(0, "split", sname)
    report_frames.append(df_rpt)

combined_report = pd.concat(report_frames)

with mlflow.start_run(run_id=RUN_ID):
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "classification_report.csv")
        combined_report.to_csv(path)
        mlflow.log_artifact(path, "reports")

## 9b — Per-class Prediction Quality

In [ ]:
from sklearn.preprocessing import label_binarize

_classes = list(CLASS_LABELS.keys())   # [0, 1, 2]
_splits  = [("val", X_val, y_val), ("test", X_test, y_test)]
_clr     = {"val": "steelblue", "test": "darkorange"}

# ── Per-class ROC curves (One-vs-Rest) ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for sname, X, y in _splits:
    proba = model.predict_proba(X)
    y_bin = label_binarize(y, classes=_classes)
    for ci in range(3):
        auc = roc_auc_score(y_bin[:, ci], proba[:, ci])
        fpr, tpr, _ = roc_curve(y_bin[:, ci], proba[:, ci])
        axes[ci].plot(fpr, tpr, color=_clr[sname], lw=2,
                      label=f"{sname}  AUC={auc:.3f}")

for ci, cls in enumerate(_classes):
    axes[ci].plot([0, 1], [0, 1], "k:", lw=1)
    axes[ci].set_title(f"Class {cls}: {CLASS_LABELS[cls]} (OvR)")
    axes[ci].set_xlabel("FPR")
    axes[ci].set_ylabel("TPR")
    axes[ci].legend(fontsize=9)

fig.suptitle(f"Per-Class ROC Curves (One-vs-Rest) + {ENABLED_NN_LABEL}", fontsize=13)
plt.tight_layout()

with mlflow.start_run(run_id=RUN_ID):
    with tempfile.TemporaryDirectory() as tmp:
        _p = os.path.join(tmp, "per_class_roc.png")
        fig.savefig(_p)
        mlflow.log_artifact(_p, "plots")
plt.show()

# ── Per-class Precision / Recall / F1 bar chart ───────────────────────────────
_x      = np.arange(3)
_w      = 0.35
_labels = [f"{CLASS_LABELS[c]}\n({c})" for c in _classes]

fig2, axes2 = plt.subplots(1, 3, figsize=(14, 5), sharey=True)
for ax, metric_fn, title in zip(
    axes2,
    [precision_score, recall_score, f1_score],
    ["Precision", "Recall", "F1"],
):
    for offset, (sname, X, y) in zip([-_w / 2, _w / 2], _splits):
        scores = metric_fn(y, model.predict(X), average=None, zero_division=0)
        ax.bar(_x + offset, scores, _w, label=sname, color=_clr[sname], alpha=0.85)
    ax.set_xticks(_x)
    ax.set_xticklabels(_labels)
    ax.set_ylim(0, 1.05)
    ax.set_title(title)
    ax.legend(fontsize=9)

fig2.suptitle(f"Per-Class Precision / Recall / F1  (val vs test) + {ENABLED_NN_LABEL}", fontsize=13)
plt.tight_layout()

with mlflow.start_run(run_id=RUN_ID):
    with tempfile.TemporaryDirectory() as tmp:
        _p2 = os.path.join(tmp, "per_class_prf1.png")
        fig2.savefig(_p2)
        mlflow.log_artifact(_p2, "plots")
plt.show()

## 10 — Analysis: Learning Curves (Overfitting Diagnostic)

In [ ]:
evals = model.evals_result()
train_logloss = evals["validation_0"]["mlogloss"]
val_logloss   = evals["validation_1"]["mlogloss"]
rounds = range(1, len(train_logloss) + 1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(rounds, train_logloss, label="train logloss", color="steelblue", lw=1.5)
ax.plot(rounds, val_logloss,   label="val logloss",   color="darkorange", lw=1.5)
ax.axvline(model.best_iteration, ls='--', color='seagreen', lw=1.5,
           label=f"best iter = {model.best_iteration}")

if model.best_iteration < len(train_logloss):
    ax.axvspan(model.best_iteration, len(train_logloss), alpha=0.08, color='red',
               label="overfitting region")

ax.set_xlabel("Boosting Round")
ax.set_ylabel("Log Loss")
ax.set_title(f"Learning Curves — {DATA_CFG['pair']} {DATA_CFG['timeframe']} + {ENABLED_NN_LABEL}")
ax.legend()
plt.tight_layout()

overfit_gap = val_logloss[model.best_iteration - 1] - train_logloss[model.best_iteration - 1]
with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_metric("overfit_logloss_gap", overfit_gap)
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "learning_curves.png")
        fig.savefig(path)
        mlflow.log_artifact(path, "plots")

print(f"Val-Train logloss gap at best iter: {overfit_gap:.6f}")
plt.show()

## 11 — Analysis: Feature Importance (Gain / Weight / Cover)

Pipeline features are shown in blue; enabled NN providers use separate colors.


In [ ]:
importance_types = ["gain", "weight", "cover"]
imp_frames = {}

for imp_type in importance_types:
    scores = model.get_booster().get_score(importance_type=imp_type)
    df_imp = pd.Series(scores, name=imp_type).reindex(feature_cols_ext).fillna(0)
    df_imp = df_imp.sort_values(ascending=False)
    imp_frames[imp_type] = df_imp

imp_combined = pd.DataFrame(imp_frames)
imp_combined.index.name = "feature"

provider_colors = {
    "pipeline": "steelblue",
    "chronos":  "darkorange",
    "kronos":   "seagreen",
    "timesfm":  "mediumpurple",
}

def _feature_color(feature: str) -> str:
    return provider_colors.get(nn_feature_provider_by_col.get(feature, "pipeline"), "gray")

fig, axes = plt.subplots(1, 3, figsize=(16, 7))
for ax, imp_type in zip(axes, importance_types):
    top20 = imp_frames[imp_type].head(20)
    top20.plot.barh(ax=ax, color=[_feature_color(f) for f in top20.index])
    ax.invert_yaxis()
    ax.set_title(f"Importance: {imp_type}")
    ax.set_xlabel("Score")

from matplotlib.patches import Patch
legend_els = [Patch(facecolor=provider_colors["pipeline"], label="Pipeline features")]
legend_els += [
    Patch(facecolor=provider_colors[source], label=f"{source} features")
    for source in nn_feature_frames.keys()
]
axes[1].legend(handles=legend_els, loc="lower right", fontsize=9)

plt.suptitle(
    f"Feature Importance — {DATA_CFG['pair']} {DATA_CFG['timeframe']} + {ENABLED_NN_LABEL}",
    y=1.01,
)
plt.tight_layout()

with mlflow.start_run(run_id=RUN_ID):
    with tempfile.TemporaryDirectory() as tmp:
        img_path = os.path.join(tmp, "feature_importance.png")
        csv_path = os.path.join(tmp, "feature_importance.csv")
        fig.savefig(img_path)
        imp_combined.to_csv(csv_path)
        mlflow.log_artifact(img_path, "plots")
        mlflow.log_artifact(csv_path, "reports")

plt.show()
print("\nTop-10 features by gain:")
imp_combined["gain"].sort_values(ascending=False).head(10)


## 12 — Analysis: Calibration Curve (Reliability Diagram)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0, 1], [0, 1], 'k--', lw=1, label="perfect calibration")

brier_scores = {}
for sname, X, y in [("val", X_val, y_val), ("test", X_test, y_test)]:
    proba_long = model.predict_proba(X)[:, 2]
    y_long     = (y == 2).astype(int)
    frac_pos, mean_pred = calibration_curve(y_long, proba_long, n_bins=10, strategy="uniform")
    brier      = brier_score_loss(y_long, proba_long)
    brier_scores[sname] = brier
    ax.plot(mean_pred, frac_pos, 's-', color=colors[sname], lw=2,
            label=f"{sname}  Brier = {brier:.4f}")

ax.set_xlabel("Mean Predicted P(long hit)")
ax.set_ylabel("Fraction of Positives")
ax.set_title(f"Calibration Curve (long hit) — {DATA_CFG['pair']} {DATA_CFG['timeframe']} + {ENABLED_NN_LABEL}")
ax.legend()
plt.tight_layout()

with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_metrics({f"{k}_brier": v for k, v in brier_scores.items()})
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "calibration_curve.png")
        fig.savefig(path)
        mlflow.log_artifact(path, "plots")

plt.show()

## 13 — Analysis: Metrics Comparison (Train / Val / Test)

In [ ]:
metrics_df = pd.DataFrame(split_metrics).T
metrics_df.index.name = "split"

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
plot_metrics = [
    (["auc", "auc_long", "avg_precision", "balanced_acc"], "Discrimination"),
    (["f1", "precision", "recall"],                        "Classification (macro)"),
    (["logloss", "brier_long"],                            "Calibration (lower = better)"),
]

for ax, (cols, title) in zip(axes, plot_metrics):
    metrics_df[cols].plot.bar(ax=ax, rot=0)
    ax.set_title(title)
    ax.set_ylim(0)
    ax.legend(fontsize=8)

plt.suptitle(f"Train / Val / Test Metric Comparison — + {ENABLED_NN_LABEL}", y=1.02)
plt.tight_layout()

with mlflow.start_run(run_id=RUN_ID):
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "metrics_comparison.png")
        fig.savefig(path)
        mlflow.log_artifact(path, "plots")

plt.show()
print(metrics_df.round(4).to_string())

## 14 — Run Summary

In [ ]:
required_summary_vars = [
    "RUN_ID", "split_metrics", "EXPERIMENT_NAME", "MLFLOW_DB", "DATA_CFG", "pd",
    "nn_feature_paths", "feature_cols", "nn_feature_cols", "nn_feature_cols_by_source",
    "feature_cols_ext",
]
missing_summary_vars = [name for name in required_summary_vars if name not in globals()]

if missing_summary_vars:
    print("Run summary is not available yet.")
    print("Missing variables: " + ", ".join(missing_summary_vars))
    print("Run the config, NN feature join, and training/MLflow cells first, then rerun the analysis cells.")
else:
    print("=" * 60)
    print(f"  MLflow Run ID  : {RUN_ID}")
    print(f"  Experiment     : {EXPERIMENT_NAME}")
    print(f"  Tracking DB    : {MLFLOW_DB}")
    print(f"  Primary run    : {DATA_CFG['norm_method']} (d={DATA_CFG.get('fracdiff_d')})")
    print("=" * 60)
    print("  To browse the UI:")
    print(f"  mlflow ui --backend-store-uri {MLFLOW_DB}")
    print("  then open  http://127.0.0.1:5000")
    print("=" * 60)
    print()
    print("NN feature parquets:")
    for source, meta in nn_feature_paths.items():
        print(f"  {source:8s}: {meta['path']}")
    print(f"Pipeline features: {len(feature_cols)}")
    print(f"NN features      : {len(nn_feature_cols)}")
    for source, cols in nn_feature_cols_by_source.items():
        print(f"  {source:8s}: {len(cols)}  {cols}")
    print(f"Total features   : {len(feature_cols_ext)}")
    print()
    print("Logged artifacts:")
    for artifact in [
        "plots/roc_curve.png",
        "plots/pr_curve.png",
        "plots/confusion_matrices.png",
        "plots/learning_curves.png",
        "plots/feature_importance.png",
        "plots/calibration_curve.png",
        "plots/metrics_comparison.png",
        "reports/classification_report.csv",
        "reports/feature_importance.csv",
        "features.json",
        "model/  (XGBoost model)",
    ]:
        print(f"  {artifact}")

    print()
    test_metrics = split_metrics.get("test", {})
    if test_metrics:
        print("Test metrics:")
        print(pd.Series(test_metrics).round(4).to_string())
    else:
        print("No test metrics were found in split_metrics.")
